# EES API Workflow

The following article presents an example workflow of how, as an analyst, someone might
connect to the Explore Education Statistics (EES) API to collect data with the help of
`eesyapi` in Python.

As an overview, the stages you will likely need to follow are:

- Find the data set ID
- Query the data set meta data to identify filter and indicator IDs
- Design a query using the relevant IDs
- Post the query to the API
- Perform some post processing of the data

In [ ]:
from get_publications import get_publications
from get_data_catalogue import get_data_catalogue
from get_meta import get_meta
from preview_dataset import preview_dataset
from query_dataset import query_dataset
from examples import example_id

## Finding a Data Set ID

Only some data sets on EES are available via the EES API. This is to maintain
minimum standards around the API data catalogue.

Assuming you know the data set within a publication that you would like to query,
you will first need to identify the data set ID code. There are two methods to do
this, either:
- **a)** via the data set's page on the EES website, or
- **b)** by querying the API itself.

### Identifying a Data Set ID via the API

Finding a data set ID via the API is a two step process:
1. Find the parent publication ID
2. Search the data catalogue within that publication for the data set

**Step 1:** Query the publication list:

In [ ]:
publications = get_publications(ees_environment="prod")
publications

This returns a list of all publications by ID and title (along with slug, summary
and date last published). From this list, you can find the title of the publication
that you're interested in and pick out the associated publication ID.

For example, we can pick out **"Pupil attendance in schools"**, which has an ID of
`cbbd299f-8297-44bc-92ac-558bcf51f8ad`.

**Step 2:** Use this publication ID to find all data sets within that publication:

In [ ]:
publication_id = "cbbd299f-8297-44bc-92ac-558bcf51f8ad"

catalogue = get_data_catalogue(
    publication_id=publication_id,
    ees_environment="prod"
)
catalogue

This returns a list containing data set IDs and titles (along with a summary,
status and information on the latest version). The data set IDs can be used as
shown in the following sections.

### Finding a Data Set ID on the EES Website

You can also find an API data set in the EES data catalogue at:
[https://explore-education-statistics.service.gov.uk/data-catalogue](https://explore-education-statistics.service.gov.uk/data-catalogue)

Filter to **"API data sets only"** to see only datasets available via the API.
Once you find your dataset, click on the title to see the **API data set ID**
in the **API data set quick start** section.

## Preview Underlying Data

In the EES API you have the option to preview the underlying data for a given
data set. This bypasses the use of IDs, making it a simpler short-term option
to explore the available data.

Using `eesyapi`, you can preview a data set using `preview_dataset()` and
providing the data set ID:

In [ ]:
dataset_id = "63629501-d3ca-c471-9780-ec4cb6fdf172"

preview = preview_dataset(
    dataset_id=dataset_id,
    ees_environment="prod",
    n_max=10
)
preview

This returns a DataFrame containing the first 10 rows of the data in the form
that DfE analysts uploaded it to EES.

> ** Warning:** While you can set `n_max=float('inf')` to get the whole
> dataset, you should avoid relying on `preview_dataset()` in your pipelines
> or dashboards. Instead use it to explore the data as you create your own
> custom query using `query_dataset()`.
>
> If you only read the whole file using the preview you risk:
> - Code breaking whenever element labels change
> - Reading in too much data into active memory, causing Python to slow or crash

## Getting the Data Set Meta Data

When querying a data set via the API, column names (indicators and filters) and
the options available within columns (filter items) are referenced using
auto-generated ID codes (known as **SQIDs**).

Therefore, in order to construct a query that involves selecting a subset of
filter items or indicators, you'll need to obtain the IDs associated with
those elements.

All those IDs are stored in a data set's meta data, which can be queried via
the API using `get_meta()`:

In [ ]:
dataset_id = "63629501-d3ca-c471-9780-ec4cb6fdf172"

meta = get_meta(
    dataset_id=dataset_id,
    ees_environment="prod"
)

print(list(meta.keys()))

This function returns a dictionary of 5 DataFrames:

- **`time_periods`**: label, code and period of all available time periods
- **`locations`**: geographic_level, code, label and item_id of all available locations
- **`filter_columns`**: col_name and label of all filters in the data set
- **`filter_items`**: col_name, item_label, item_id and default_item for all filter items
- **`indicators`**: col_name and label of all indicators in the data set

The `item_id` values in `locations` and `filter_items` are required to construct
queries for filtering a given data set.

### Time Periods

In [ ]:
time_periods = meta["time_periods"]
time_periods[time_periods["label"].isin(["2025 Week 4", "2025 Week 5"])]

### Geographies

In [ ]:
locations = meta["locations"]
locations[locations["label"].isin(["York", "England"])]

### Filter Columns

Note: the `col_id` values here correspond to the `col_id` values in filter items,
which shows which column each item is available in.

In [ ]:
meta["filter_columns"]

### Filter Items

In [ ]:
filter_items = meta["filter_items"]
filter_items[filter_items["item_label"].isin([
    "Absence",
    "Approved educational activity",
    "Authorised",
    "Unauthorised"
])]

### Indicators

In [ ]:
indicators = meta["indicators"]
indicators[indicators["label"].isin(["Number of sessions"])]

## Retrieving Data from a Data Set

### Using `query_dataset()`

The recommended option for retrieving data is the `query_dataset()` function.
This provides options for:
- Attaching a pre-made JSON dict to a query
- Passing a set of IDs as parameters to create a query for you

#### Querying a Data Set Using Parameters

`query_dataset()` can be given a set of parameters to build up a query.
The parameters you can provide are:

- `time_periods`
- `geographies`
- `filter_items`
- `indicators` **(required)**

All are optional except `indicators`. If you don't provide any of `time_periods`,
`geographies` or `filter_items`, you'll receive all rows from the data set
(be careful — this may take some time for larger data sets).

In [ ]:
result = query_dataset(
    dataset_id="63629501-d3ca-c471-9780-ec4cb6fdf172",
    indicators="X9fKb",
    filter_items={
        "attendance_status": ["e4wuS"],
        "attendance_type": ["P9Aeb", "1Dui3", "oRfmX"],
        "time_frame": ["5ezdi"]
    },
    geographies=["NAT|id|dP0Zw", "LA|id|BT7J3"],
    time_periods=["2025|W4"],
    page_size=15,
    page=1,
    ees_environment="prod"
)
result

Some notes on the above:

- `indicators`, `time_periods`, `geographies` and `filter_items` can all be
  supplied as lists and will be interpreted as `filter(... isin([...]))`.
- `filter_items` can be supplied as a **dict** to create more stringent
  cross-filter combinations — rows will only be returned for the specific
  combinations given across the filters.
- `geographies` can be supplied as a DataFrame to allow for more complex
  queries such as "give me all LAs in a given region".

#### Querying a Data Set Using a JSON Dict

If you're familiar with JSON and wish to write your own query from scratch,
you can use `query_dataset()` with the `json_query` parameter:

In [ ]:
json_query = {
    "criteria": {
        "and": [
            {
                "filters": {
                    "eq": "3xu8u"
                }
            },
            {
                "locations": {
                    "eq": {
                        "level": "LA",
                        "id": "BT7J3"
                    }
                }
            }
        ]
    },
    "indicators": ["X9fKb"],
    "debug": False,
    "page": 1,
    "pageSize": 10
}

result = query_dataset(
    dataset_id="63629501-d3ca-c471-9780-ec4cb6fdf172",
    json_query=json_query,
    ees_environment="prod"
)
result

## FAQs

### Why do you use IDs rather than the actual labels used in the data?

The IDs are intended to aid in future proofing for users that may develop
automated pipelines that connect to the EES API. Labelling inevitably changes —
for example if a "Total" entry is renamed to "All pupils" to add clarity.

If the underlying content stays the same, then automated pipelines will still
need to connect to the same item. Therefore each item has an associated ID
that will remain unchanged in a given data set as it is updated with new data
over subsequent releases, even if a renaming of a field occurs.

### Are there any bandwidth / repeat connection constraints?

There are currently no constraints whilst the system is being tested.